# SAE 3.1 — Flux Média projet "CodecPlayer"

**Étudiants :**  
- GIUFFRE Matteo  
- GALES Quentin  

---

## Introduction

Dans le cadre de la SAE 3.1, nous avons réalisé un outil de gestion et de traitement de flux multimédia en utilisant **Python**, **ipywidgets** et **GStreamer**.

L’objectif de ce projet était de concevoir une interface interactive permettant de :
- Lire des fichiers audio (MP3, WAV, FLAC)
- Afficher une représentation visuelle du signal audio
- Afficher une pochette associée au média
- Extraire et afficher les informations du fichier
- Gérer l’affichage de sous-titres lorsque ceux-ci sont disponibles
- Convertir et exporter les fichiers audio avec différents paramètres (format, fréquence d’échantillonnage, quantification)

Le projet est structuré en plusieurs parties correspondant aux différents besoins fonctionnels demandés dans l’énoncé.  
Ce document présente le compte rendu des **Séances Non-encadrée (NE) — Parties 1, 2 et 3**.

---


In [1]:
pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [3]:
import ipywidgets as widgets
from ipywidgets import Layout, Checkbox, Box, AppLayout, Button
import subprocess
from IPython.display import display
from pathlib import Path

PATH = "/home/users/etudiant/g/gq403784/SAE31-3-4-5/Fichiers_Utiles_Projet"

fic_audio = widgets.FileUpload(
    accept='.mp3,.flac,.wav',
    multiple=False,
    description='Selectionner le fichier',
    layout=Layout(width='100%'),
    button_style='primary'
)

items_layout = Layout(width='auto')
box_layout = Layout(display='flex', flex_flow='row', align_items='center', gap='20px',left='6%')  
words = ['Representation visuelle', 'Pochette', 'Informations','Sous-Titres']
items = [Checkbox(value=True, description=word, layout=items_layout) for word in words]
box = Box(children=items, layout=box_layout)
visuel, pochette, meta_donnees, sous_titres = items 


play = widgets.Button(
    description='Play',
    icon='play',
    button_style='success'
)

stop = widgets.Button(
    description='Stop',
    icon='stop',
    button_style='danger'
)

buttons = Box(
    children=[play, stop],
    layout=Layout(display='flex', gap='20px', justify_content='center')
)


output_part1 = widgets.Output()  


def stop_clique(b=None):
    subprocess.run("pkill -f gst-launch-1.0", shell=True)
    with output_part1:
        output_part1.clear_output()
        print("Lecture stoppée")


def _on_sous_titres_change(change):  
    if change["name"] == "value" and change["new"] is True:
        if not pochette.value:
            pochette.value = True

def _on_pochette_change(change): 
    if change["name"] == "value" and change["new"] is False:
        if sous_titres.value:
            sous_titres.value = False

sous_titres.observe(_on_sous_titres_change, names="value") 
pochette.observe(_on_pochette_change, names="value")      

def btn_clique(b):
    with output_part1:
        output_part1.clear_output()

        if not fic_audio.value:
            print("Vous n'avez pas séléctionné de fichier audio !")
            return

        uploaded = fic_audio.value[0]
        fic_selectionne = uploaded["name"]
        filepath = f"{PATH}/{fic_selectionne}"
        stem = Path(uploaded["name"]).stem
        coverpath = f"{PATH}/{stem}.jpg"
        srtpath = f"{PATH}/{stem}.srt"

        cover_exists = Path(coverpath).exists()

        if sous_titres.value:
            if Path(srtpath).exists():
                cmd_sub = (
                    f"gst-launch-1.0 -e -v "
                    f"filesrc location='{filepath}' ! decodebin name=d "
                    f"d. ! queue ! audioconvert ! audioresample ! autoaudiosink "
                    f"filesrc location='{coverpath}' ! jpegparse ! jpegdec ! imagefreeze ! videoconvert "
                    f"! subtitleoverlay name=ov ! videoconvert ! autovideosink "
                    f"filesrc location='{srtpath}' ! subparse ! queue ! ov."
                )
                subprocess.Popen(cmd_sub, shell=True)

                if visuel.value:
                    cmd_visu = (
                        f"gst-launch-1.0 -e -v "
                        f"filesrc location='{filepath}' ! decodebin ! audioconvert "
                        f"! wavescope ! videoconvert ! autovideosink"
                    )
                    subprocess.Popen(cmd_visu, shell=True)

                if meta_donnees.value:
                    subprocess.run(f"gst-discoverer-1.0 '{filepath}'", shell=True)
                return
            else:
                print(f" Sous-titres cochés mais fichier introuvable : {srtpath}")

        if visuel.value and pochette.value and meta_donnees.value:
            cmd_meta = f"gst-discoverer-1.0 '{filepath}'"

            if cover_exists:
                cmd_visuel = f"gst-launch-1.0 -v filesrc location='{filepath}' ! decodebin ! audioconvert ! tee name=t t. ! queue ! wavescope ! videoconvert ! autovideosink t. ! queue ! audioresample ! autoaudiosink"
                cmd_pochette = f"gst-launch-1.0 -v filesrc location='{coverpath}' ! jpegparse ! jpegdec ! imagefreeze ! videoconvert ! autovideosink"
                subprocess.Popen(cmd_visuel, shell=True)
                subprocess.Popen(cmd_pochette, shell=True)
            else:
                print("Il n'y a pas de pochette pour ce titre...")
                cmd_visuel = f"gst-launch-1.0 -v filesrc location='{filepath}' ! decodebin ! audioconvert ! tee name=t t. ! queue ! wavescope ! videoconvert ! autovideosink t. ! queue ! audioresample ! autoaudiosink"
                subprocess.Popen(cmd_visuel, shell=True)
                subprocess.run(cmd_meta, shell=True)

        elif visuel.value and pochette.value:
            cmd_visuel = f"gst-launch-1.0 -v filesrc location='{filepath}' ! decodebin ! audioconvert ! tee name=t t. ! queue ! wavescope ! videoconvert ! autovideosink t. ! queue ! audioresample ! autoaudiosink"
            cmd_pochette = f"gst-launch-1.0 -v filesrc location='{coverpath}' ! jpegparse ! jpegdec ! imagefreeze ! videoconvert ! autovideosink"
            subprocess.Popen(cmd_visuel, shell=True)
            subprocess.run(cmd_pochette, shell=True)

        elif visuel.value and meta_donnees.value:
            cmd_visuel = f"gst-launch-1.0 -v filesrc location='{filepath}' ! decodebin ! audioconvert ! tee name=t t. ! queue ! wavescope ! videoconvert ! autovideosink t. ! queue ! audioresample ! autoaudiosink"
            cmd_meta = f"gst-discoverer-1.0 '{filepath}'"
            subprocess.Popen(cmd_visuel, shell=True)
            subprocess.run(cmd_meta, shell=True)

        elif pochette.value and meta_donnees.value:
            cmd_pochette = f"gst-launch-1.0 -v filesrc location='{coverpath}' ! jpegparse ! jpegdec ! imagefreeze ! videoconvert ! autovideosink"
            cmd_audio = f"gst-launch-1.0 -v filesrc location='{filepath}' ! decodebin ! audioconvert ! audioresample ! autoaudiosink"
            cmd_meta = f"gst-discoverer-1.0 '{filepath}'"
            subprocess.Popen(cmd_pochette, shell=True)
            subprocess.Popen(cmd_audio, shell=True)
            subprocess.run(cmd_meta, shell=True)

        elif visuel.value:
            cmd = f"gst-launch-1.0 -v filesrc location={filepath} ! decodebin ! audioconvert ! tee name=t t. ! queue ! wavescope ! videoconvert ! autovideosink t. ! queue ! audioresample ! autoaudiosink"
            subprocess.Popen(cmd, shell=True)

        elif pochette.value:
            if cover_exists:
                cmd_pochette = f"gst-launch-1.0 -v filesrc location='{coverpath}' ! jpegparse ! jpegdec ! imagefreeze ! videoconvert ! autovideosink"
                subprocess.Popen(cmd_pochette, shell=True)
            else:
                print("Il n'y a pas de pochette pour ce titre...")

            cmd_audio = f"gst-launch-1.0 -v filesrc location='{filepath}' ! decodebin ! audioconvert ! audioresample ! autoaudiosink"
            subprocess.Popen(cmd_audio, shell=True)

        elif meta_donnees.value:
            cmd_audio = f"gst-launch-1.0 filesrc location='{filepath}' ! decodebin ! audioconvert ! audioresample ! autoaudiosink"
            cmd_meta = f"gst-discoverer-1.0 '{filepath}'"
            subprocess.Popen(cmd_audio, shell=True)
            subprocess.run(cmd_meta, shell=True)

        else:
            cmd = f"gst-launch-1.0 -v filesrc location='{filepath}' ! decodebin ! audioconvert ! audioresample ! autoaudiosink"
            subprocess.Popen(cmd, shell=True)


play.on_click(btn_clique)
stop.on_click(stop_clique)

app1 = AppLayout(
    header=fic_audio,
    left_sidebar=None,
    center=box,
    right_sidebar=None,
    footer=buttons,
    pane_widths=[0, 1, 0],
    pane_heights=[0, 2, 1],
    layout=Layout(border='2px solid black',padding='10px',width='auto')
)



format_choisi = widgets.Dropdown(
    options=[("MP3", "mp3"), ("WAV", "wav"), ("FLAC", "flac")],
    description="Format :",
    layout=Layout(width='250px')
)


rate = widgets.IntSlider(
    value=44100,
    min=8000,
    max=48000,
    step=1,
    description="Rate :",
    layout=Layout(width='350px')
)

quantification = widgets.Dropdown(
    options=[("16 bits", 16), ("24 bits", 24), ("32 bits", 32)],
    value=16,
    description="Quantif :",
    layout=Layout(width='250px')
)

export = widgets.Checkbox(
    value=False,
    description='Export'
)

play2 = widgets.Checkbox(
    value=False,
    description='Play'
)

go = widgets.Button(
    description='Go !',
    icon='check',
    button_style='success',
    layout=Layout(left='43%')
)

output_part2 = widgets.Output() 

Prem_ligne = Box(
    children=[format_choisi, rate, quantification],
    layout=Layout(display='flex', flex_flow='row', align_items='center', gap='20px', margin='20px')
)

Deux_ligne = Box(
    children=[export, play2],
    layout=Layout(display='flex', flex_flow='row', align_items='center', gap='40px', margin='10px', left='13%')
)

options_box = Box(
    children=[Prem_ligne, Deux_ligne],
    layout=Layout(display='flex', flex_flow='column', align_items='flex-start', gap='10px', left='10%')
)

def play_fct(path_entre, rate_val, bits):
    if bits == 16:
        gst_bits = "S16LE"
    elif bits == 24:
        gst_bits = "S24LE"
    elif bits == 32:
        gst_bits = "S32LE"

    cmd_play = (
        f"gst-launch-1.0 -v filesrc location='{path_entre}' ! decodebin ! audioconvert ! audioresample ! audio/x-raw,rate={rate_val},format={gst_bits} ! autoaudiosink"
    )

    subprocess.Popen(cmd_play, shell=True)

def export_fct():
    with output_part2:
        output_part2.clear_output() 

        if not export.value:
            print("T'as pas coché export")
            return

        if not fic_audio.value:
            print("Sélectionner un fichier audio")
            return

        uploaded = fic_audio.value[0]
        filename = uploaded["name"]
        path_entre = f"{PATH}/{filename}"

        fmt = format_choisi.value
        rate_val = int(rate.value)
        bits = int(quantification.value)

        if bits == 16:
            gst_bits = "S16LE"
        elif bits == 24:
            gst_bits = "S24LE"
        elif bits == 32:
            gst_bits = "S32LE"

        path_nv = f"{PATH}/{Path(filename).stem}_{rate_val}Hz_{bits}bit.{fmt}"

        cmd = (
            f"gst-launch-1.0 -e -v filesrc location='{path_entre}' ! decodebin ! audioconvert ! audioresample ! audio/x-raw,rate={rate_val},format={gst_bits}"
        )

        if fmt == "wav":
            cmd += "! wavenc "
        elif fmt == "flac":
            cmd += "! flacenc "
        elif fmt == "mp3":
            cmd += "! lamemp3enc bitrate=192 cbr=true ! id3v2mux "

        cmd += f"! filesink location='{path_nv}'"

        subprocess.run(cmd, shell=True)
        print("Export réussi :", path_nv)

def go_click(b):
    with output_part2:  
        output_part2.clear_output()  

        if not fic_audio.value:
            print("Sélectionner un fichier audio")
            return

        uploaded = fic_audio.value[0]
        filename = uploaded["name"]
        path_entre = f"{PATH}/{filename}"

        rate_val = int(rate.value)
        bits = int(quantification.value)

        if export.value:
            export_fct()

        if play2.value:
            play_fct(path_entre, rate_val, bits)

        if not export.value and not play2.value:
            print("Coche une case")

go.on_click(go_click)

app2 = AppLayout(
    header=None,
    center=options_box,
    footer=go,
    pane_widths=[0, 1, 0],
    pane_heights=[0, 2, 1],
    layout=Layout(border='2px solid black', padding='10px', width='auto')
)

titre = widgets.HTML(
    value="<h2 style='text-align:center; margin-bottom:20px;'>Flux Média</h2>"
)

gap = widgets.Box(layout=Layout(height='10px'))

display(titre)
display(app1)
display(output_part1) 
display(gap)
display(app2)
display(output_part2)  


HTML(value="<h2 style='text-align:center; margin-bottom:20px;'>Flux Média</h2>")

AppLayout(children=(FileUpload(value=(), accept='.mp3,.flac,.wav', button_style='primary', description='Select…

Output()

Box(layout=Layout(height='10px'))

AppLayout(children=(Button(button_style='success', description='Go !', icon='check', layout=Layout(grid_area='…

Output()